# 03 — Inference Analysis

Load a trained checkpoint, run rollout on the test set, and analyse prediction quality quantitatively.

In [ ]:
import sys
import os
sys.path.insert(0, '..')  # project root

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import tri as mtri

# Load config manually (no Hydra chdir in notebooks)
from omegaconf import OmegaConf
cfg = OmegaConf.load('../conf/config.yaml')
# Remove hydra key — not needed outside CLI
cfg = OmegaConf.masked_copy(cfg, [k for k in cfg if k != 'hydra'])

# Override paths to be absolute
cfg.data_dir = '../raw_dataset/cylinder_flow/cylinder_flow'
cfg.ckpt_path = '../checkpoints'

print(OmegaConf.to_yaml(cfg))

In [ ]:
from physicsnemo.utils.logging import PythonLogger
from inference import MGNRollout

logger = PythonLogger('notebook')
rollout = MGNRollout(cfg, logger)
rollout.predict()
print(f'Predicted {len(rollout.pred)} time steps')

In [ ]:
from src.analysis.metrics import compute_metrics_report

report = compute_metrics_report(rollout.pred, rollout.exact)

print(f"{'Field':<8} {'RMSE':>10} {'Rel. Error':>12}")
print('-' * 32)
for field in ['u', 'v', 'p']:
    print(f"{field:<8} {report[f'rmse_{field}']:>10.6f} {report[f'rel_err_{field}']:>12.6f}")

In [ ]:
# Temporal RMSE — how error accumulates over time steps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, field in zip(axes, ['u', 'v', 'p']):
    temporal = report[f'temporal_rmse_{field}']
    ax.plot(temporal, linewidth=1.2)
    ax.set_title(f'{field}  RMSE={report[f"rmse_{field}"]:.4f}')
    ax.set_xlabel('Time step')
    ax.set_ylabel('RMSE')
    ax.grid(True, alpha=0.3)
plt.suptitle('Temporal RMSE (error accumulation over rollout)')
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: prediction vs ground truth at a chosen time step
T = 100  # time step index to visualise
FIELD = 0  # 0=u, 1=v, 2=p
FIELD_NAME = ['u', 'v', 'p'][FIELD]

graph = rollout.graphs[T]
cells = rollout.faces[T]
pred_f = rollout.pred[T][:, FIELD].numpy()
exact_f = rollout.exact[T][:, FIELD].numpy()

pos = graph['mesh_pos'].numpy()
triang = mtri.Triangulation(pos[:, 0], pos[:, 1], cells)

vmin = min(pred_f.min(), exact_f.min())
vmax = max(pred_f.max(), exact_f.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, data, title in zip(axes, [pred_f, exact_f], ['Prediction', 'Ground Truth']):
    tc = ax.tripcolor(triang, data, vmin=vmin, vmax=vmax, cmap='RdBu_r')
    plt.colorbar(tc, ax=ax)
    ax.set_title(f'{title} — {FIELD_NAME} at t={T}')
    ax.set_aspect('equal')
    ax.set_axis_off()
plt.tight_layout()
plt.show()